# Preprocessing Pipeline

### 1) Parse the annotation file

In [8]:
import scipy.io
import pandas as pd

# Load annotations and metadata
anno = scipy.io.loadmat('car_devkit/devkit/cars_train_annos.mat')
meta = scipy.io.loadmat('car_devkit/devkit/cars_meta.mat')

annotations = anno['annotations'][0]
class_names = [c[0] for c in meta['class_names'][0]]

parsed_data = []
for a in annotations:
    bbox = {
        'file_name': a[5][0],
        'xmin': a[0][0][0],
        'ymin': a[1][0][0],
        'xmax': a[2][0][0],
        'ymax': a[3][0][0],
        'class_id': a[4][0][0] - 1,
        'class_name': class_names[a[4][0][0] - 1]
    }
    parsed_data.append(bbox)

df = pd.DataFrame(parsed_data)
print(df.head())

   file_name  xmin  ymin  xmax  ymax  class_id  \
0  00001.jpg    39   116   569   375        13   
1  00002.jpg    36   116   868   587         2   
2  00003.jpg    85   109   601   381        90   
3  00004.jpg   621   393  1484  1096       133   
4  00005.jpg    14    36   133    99       105   

                            class_name  
0                  Audi TTS Coupe 2012  
1                  Acura TL Sedan 2012  
2           Dodge Dakota Club Cab 2007  
3     Hyundai Sonata Hybrid Sedan 2012  
4  Ford F-450 Super Duty Crew Cab 2012  


### 2) Split Dataset (80% train, 20% val)

In [9]:
import os
import shutil
from sklearn.model_selection import train_test_split

# Split
train_df, val_df = train_test_split(df, test_size=0.2, stratify=df['class_id'], random_state=42)

# Ensure dirs
os.makedirs('projects/dataset/images/train', exist_ok=True)
os.makedirs('projects/dataset/images/val', exist_ok=True)
os.makedirs('projects/dataset/labels/train', exist_ok=True)
os.makedirs('projects/dataset/labels/val', exist_ok=True)

def copy_images(split_df, split):
    for _, row in split_df.iterrows():
        src = os.path.join('cars_train', 'cars_train', row['file_name'])
        dst = os.path.join('projects/dataset/images', split, row['file_name'])
        if os.path.exists(src):
            shutil.copy2(src, dst)
        else:
            print(f"❗ Missing: {src}")

copy_images(train_df, 'train')
copy_images(val_df, 'val')

print("✅ Images copied successfully!")

✅ Images copied successfully!


### 3) Create YOLOv8 labels

In [10]:
import cv2

def create_yolo_label(row, split):
    img_path = f"projects/dataset/images/{split}/{row['file_name']}"
    label_path = f"projects/dataset/labels/{split}/{row['file_name'].replace('.jpg', '.txt')}"

    if not os.path.exists(img_path):
        return

    img = cv2.imread(img_path)
    h, w = img.shape[:2]

    # Normalized YOLO format
    x_center = ((row['xmin'] + row['xmax']) / 2) / w
    y_center = ((row['ymin'] + row['ymax']) / 2) / h
    width = (row['xmax'] - row['xmin']) / w
    height = (row['ymax'] - row['ymin']) / h

    with open(label_path, 'w') as f:
        f.write(f"{row['class_id']} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}\n")

# Generate labels
for _, row in train_df.iterrows():
    create_yolo_label(row, 'train')

for _, row in val_df.iterrows():
    create_yolo_label(row, 'val')

print("✅ YOLO labels created!")

✅ YOLO labels created!


### 4) Filter 10 Classes of Cars

In [3]:
import os
import shutil

# === CONFIGURATION ===
# Original dataset paths
orig_image_dir = "projects/dataset/images"
orig_label_dir = "projects/dataset/labels"

# New filtered dataset paths
filtered_image_dir = "filtered_dataset/images"
filtered_label_dir = "filtered_dataset/labels"

splits = ["train", "val"]

# Original class IDs to keep and their new mapped IDs
class_id_map = {
    1: 0,     # Acura RL Sedan 2012
    7: 1,     # Aston Martin V8 Vantage Convertible 2012
    14: 2,    # Audi R8 Coupe 2012
    25: 3,    # BMW ActiveHybrid 5 Sedan 2012
    44: 4,    # Bugatti Veyron 16.4 Convertible 2009
    52: 5,    # Cadillac Escalade EXT Crew Cab 2007
    54: 6,    # Chevrolet Corvette Convertible 2012
    93: 7,    # Dodge Durango SUV 2012
    127: 8,   # Honda Accord Coupe 2012
    150: 9    # Lamborghini Aventador Coupe 2012
}

# === PROCESSING ===
for split in splits:
    os.makedirs(os.path.join(filtered_image_dir, split), exist_ok=True)
    os.makedirs(os.path.join(filtered_label_dir, split), exist_ok=True)

    image_split_dir = os.path.join(orig_image_dir, split)
    label_split_dir = os.path.join(orig_label_dir, split)

    for label_file in os.listdir(label_split_dir):
        label_path = os.path.join(label_split_dir, label_file)
        with open(label_path, 'r') as f:
            lines = f.readlines()

        # Filter and remap classes
        filtered_lines = []
        for line in lines:
            parts = line.strip().split()
            class_id = int(parts[0])
            if class_id in class_id_map:
                new_class_id = class_id_map[class_id]
                new_line = " ".join([str(new_class_id)] + parts[1:]) + "\n"
                filtered_lines.append(new_line)

        if filtered_lines:
            # Save new label
            new_label_path = os.path.join(filtered_label_dir, split, label_file)
            with open(new_label_path, 'w') as f:
                f.writelines(filtered_lines)

            # Copy image
            img_name = label_file.replace('.txt', '.jpg')
            src_img_path = os.path.join(image_split_dir, img_name)
            dst_img_path = os.path.join(filtered_image_dir, split, img_name)
            if os.path.exists(src_img_path):
                shutil.copyfile(src_img_path, dst_img_path)
            else:
                print(f"Warning: Missing image for {label_file}")

print("✅ Dataset filtering and remapping complete.")


✅ Dataset filtering and remapping complete.


### 4) Create YAML File